### Tool Constructor - Practice & Exercises

In this notebook, you will practice creating custom tools using standard class constructors with Pydantic validation schemas.

#### Part 1: Follow-Along Practice

##### Goal / Question
Construct custom utility tools using Tool and StructuredTool constructors, map Pydantic schema validation, and bind them to a tool-calling agent.

##### Sample Input
```python
"Concatenate hello and world"
```

##### Sample Output
helloworld

##### Plan / Steps
1. Define helper functions `greet_user`, `reverse_string`, and `concatenate_strings`.
2. Define `ConcatenateStringsArgs` using Pydantic `BaseModel` to structure multi-parameter string inputs.
3. Wrap simple functions using `Tool(...)` constructor.
4. Wrap complex multi-variable functions using `StructuredTool.from_function(...)`.
5. Pull prompt `hwchase17/openai-tools-agent` from hub.
6. Initialize model using `init_chat_model("llama-3.3-70b-versatile", model_provider="groq")`.
7. Create agent executor, and run.

In [ ]:
from langchain import hub
from langchain.agents import AgentExecutor, create_tool_calling_agent
from pydantic import BaseModel, Field
from langchain_core.tools import StructuredTool, Tool
from langchain.chat_models import init_chat_model

def greet_user(name: str) -> str:
    return f"Hello, {name}!"

def reverse_string(text: str) -> str:
    return text[::-1]

def concatenate_strings(a: str, b: str) -> str:
    return a + b

class ConcatenateStringsArgs(BaseModel):
    a: str = Field(description="First string")
    b: str = Field(description="Second string")

tools = [
    Tool(name="GreetUser", func=greet_user, description="Greets the user by name."),
    Tool(name="ReverseString", func=reverse_string, description="Reverses string."),
    StructuredTool.from_function(func=concatenate_strings, name="ConcatenateStrings", description="Concats two strings.", args_schema=ConcatenateStringsArgs)
]
llm = init_chat_model("llama-3.3-70b-versatile", model_provider="groq")
prompt = hub.pull("hwchase17/openai-tools-agent")
agent = create_tool_calling_agent(llm=llm, tools=tools, prompt=prompt)
executor = AgentExecutor.from_agent_and_tools(agent=agent, tools=tools, verbose=True)
print(executor.invoke({"input": "Concatenate hello and world"}))

#### Part 2: Practice Exercises

##### Exercise: Power Calculator Tool

##### Goal / Question
Create a structured tool `PowerCalculator` using `StructuredTool.from_function` that computes the power result ($a^b$) for two floats `base` and `exponent`.

##### Sample Input
```python
{"input": "Calculate 2 to the power of 10"}
```

##### Sample Output
1024.0

##### Plan / Steps
1. Define a python math function `calc_power(base: float, exponent: float)`.
2. Formulate a Pydantic class `PowerCalculatorArgs` listing the float descriptions.
3. Wrap both using `StructuredTool.from_function`.
4. Add to agent tools list and test.

In [ ]:
# TODO: Write your implementation here

<details>
<summary>💡 Click to see solution</summary>

```python
def calc_power(base: float, exponent: float) -> str:
    return str(base ** exponent)

class PowerCalculatorArgs(BaseModel):
    base: float = Field(description="The base value")
    exponent: float = Field(description="The power value")

power_tool = StructuredTool.from_function(
    func=calc_power,
    name="PowerCalculator",
    description="Useful for calculating mathematical power exponent values.",
    args_schema=PowerCalculatorArgs
)
extended_tools = tools + [power_tool]
new_agent = create_tool_calling_agent(llm=llm, tools=extended_tools, prompt=prompt)
new_executor = AgentExecutor.from_agent_and_tools(agent=new_agent, tools=extended_tools, verbose=True)
print(new_executor.invoke({"input": "Calculate 2 to the power of 10"}))
```
</details>